In [1]:
!pip install ultralytics
!pip install supervision

  Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached torch-2.12.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import cv2
import numpy as np

from ultralytics import YOLO

from collections import defaultdict

from IPython.display import Video

In [3]:
model = YOLO("yolo11n.pt")

print("Model loaded")

Model loaded


In [4]:
video_path = "D:/speed2/m6.mp4"

output_path = "D:/speed2/m6_output.mp4"

In [5]:
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

print("FPS :", fps)
print("Width :", width)
print("Height :", height)
print("Frames :", total_frames)

cap.release()

FPS : 25.0
Width : 1920
Height : 1080
Frames : 1501


In [6]:
#inisialisasi tracking
track_history = defaultdict(list)

print("Tracking history initialized")

Tracking history initialized


In [7]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

print("Output video initialized")

Output video initialized


In [8]:
import time

start_time = time.time()

print("Timer started")

Timer started


In [9]:
vehicle_ids = set()

print("Vehicle counter initialized")

Vehicle counter initialized


In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import time
from collections import defaultdict

# Pastikan path video dan model sudah terdefinisi
if 'video_path' not in globals() or 'output_path' not in globals():
    video_path = "D:/speed2/m6.mp4"
    output_path = "D:/speed2/m6_output.mp4"

if 'model' not in globals():
    from ultralytics import YOLO
    model = YOLO("yolo11n.pt")

# ==========================================
# INITIALIZATION (Self-contained & Re-runnable)
# ==========================================

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("========== INPUT ==========")
print("FPS:", fps)
print("Total Frames:", total_frames)
print("Resolution:", f"{width}x{height}")

# Reset history & counter untuk setiap kali cell dijalankan ulang
track_history = defaultdict(list)
vehicle_ids = set()

# Inisialisasi Video Writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

# Faktor kalibrasi Piksel ke Meter (Pixels Per Meter)
# Sesuaikan PPM ini dengan perspektif kamera Anda untuk mendapatkan kecepatan km/jam yang akurat.
# Jika PPM = 1.0 dan gunakan_kmh = False, maka kecepatan akan dihitung dalam piksel per detik.
PPM = 10.0          # Contoh: 10 piksel = 1 meter
gunakan_kmh = True  # Set True untuk menampilkan km/jam, False untuk piksel/detik

frame_number = 0
start_time = time.time()

print("\n========== PROCESSING VIDEO ==========")
while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # =====================
    # VEHICLE DETECTION + TRACKING
    # =====================

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[2, 3, 5, 7],  # car, motorcycle, bus, truck
        verbose=False
    )

    annotated_frame = frame.copy()

    # =====================
    # TRACKING RESULT
    # =====================

    if (
        results[0].boxes is not None
        and results[0].boxes.id is not None
    ):

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, ids):

            vehicle_ids.add(track_id)

            x1, y1, x2, y2 = map(int, box)

            # =====================
            # CENTROID EXTRACTION
            # =====================

            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            centroid = (cx, cy)

            # =====================
            # SAVE HISTORY
            # =====================

            track_history[track_id].append(
                centroid
            )

            # Batasi history agar tidak terlalu panjang
            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

            history = track_history[track_id]

            # =====================
            # SPEED ESTIMATION
            # =====================

            speed = 0

            if len(history) > 1:
                # Menggunakan rentang frame (window) agar estimasi kecepatan lebih stabil (meredam jitter)
                window_size = min(5, len(history))
                p1 = history[-window_size]
                p2 = history[-1]

                distance = np.sqrt(
                    (p2[0] - p1[0])**2 +
                    (p2[1] - p1[1])**2
                )

                # Selisih waktu dalam detik
                time_interval = (window_size - 1) / fps

                if time_interval > 0:
                    speed_px_per_sec = distance / time_interval
                    
                    if gunakan_kmh:
                        # Konversi piksel/detik ke km/jam
                        speed = (speed_px_per_sec / PPM) * 3.6
                    else:
                        speed = speed_px_per_sec

            # =====================
            # DRAW BOUNDING BOX
            # =====================

            cv2.rectangle(
                annotated_frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # =====================
            # DRAW CENTROID
            # =====================

            cv2.circle(
                annotated_frame,
                centroid,
                4,
                (0, 0, 255),
                -1
            )

            # =====================
            # LABEL
            # =====================

            unit = "km/h" if gunakan_kmh else "px/s"
            label = f"ID:{track_id} Speed:{speed:.1f} {unit}"

            (text_w, text_h), _ = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                2
            )

            cv2.rectangle(
                annotated_frame,
                (x1, y1 - text_h - 15),
                (x1 + text_w + 5, y1),
                (0, 0, 0),
                -1
            )

            cv2.putText(
                annotated_frame,
                label,
                (x1 + 2, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            # =====================
            # TRAJECTORY
            # =====================

            if len(history) > 1:

                for i in range(1, len(history)):

                    cv2.line(
                        annotated_frame,
                        history[i - 1],
                        history[i],
                        (255, 0, 0),
                        2
                    )

    cv2.putText(
        annotated_frame,
        f"Vehicles: {len(vehicle_ids)}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )

    out.write(annotated_frame)

    if frame_number % 50 == 0:

        print(
            f"Processed {frame_number}/{total_frames}"
        )

cap.release()
out.release()

print("\nProcessing Finished")

# =====================
# OUTPUT CHECK
# =====================

cap2 = cv2.VideoCapture(output_path)

output_frames = int(
    cap2.get(cv2.CAP_PROP_FRAME_COUNT)
)

output_fps = cap2.get(
    cv2.CAP_PROP_FPS
)

print("\n========== OUTPUT ==========")
print("Output FPS:", output_fps)
print("Output Frames:", output_frames)

cap2.release()

========== INPUT ==========


NameError: name 'fps' is not defined

In [ ]:
print("========== VEHICLE COUNT ==========")

print(
    "Total Vehicles Detected:",
    len(vehicle_ids)
)

In [ ]:
end_time = time.time()

processing_time = end_time - start_time

processing_fps = total_frames / processing_time

print("========== PERFORMANCE ==========")

print(
    f"Processing Time : {processing_time:.2f} sec"
)

print(
    f"Processing FPS  : {processing_fps:.2f}"
)

In [ ]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():

    print(torch.cuda.get_device_name(0))